# GRIB2 to JMV Data Converter

Made by Kai Hatton

Welcome to this interactive Google Colab notebook designed to simplify the conversion of GRIB2 meteorological data into the JMV file format. This tool is especially useful for users in maritime and meteorological domains who rely on JMV files for specialized applications, but often receive primary weather data in the GRIB2 format.

**Purpose:**

This notebook provides a **no-code-required** experience for transforming GRIB2 files (e.g., from GFS, ECMWF) into JMV files directly within your browser using Google Colab.

**How to Use:**

Simply run the cells in order from top to bottom by clicking the 'Play' button (▶) next to each cell. Follow the instructions in the markdown cells to select your data source and download your converted files.

## 1. Initial Setup

Before we can start processing GRIB2 files, we need to install a few essential libraries. These libraries handle GRIB2 parsing, data manipulation, and file operations. The cell below will install everything needed in your Colab environment.

In [ ]:
# Install Python packages
!pip install --quiet ecmwf-opendata cfgrib xarray boto3 pandas requests

# Install system libraries required by cfgrib (for GRIB2 decoding)
!apt-get install --quiet -y libeccodes-dev

print("Installation complete. You might see some warning messages, but as long as no fatal errors occurred, you can proceed.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.1/49.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.4/15.4 MB 62.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.6/91.6 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.1/90.1 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 88.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 80.0 MB/s eta 0:00:00
Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  libeccodes-data libeccodes0
The following NEW packages will be installed:
  libeccodes-data libeccodes-dev libeccodes0
0 upgraded, 3 newly installed, 0 to remove and 3 not upgraded.
Need to get 3,076 kB of archives.
After this operation

In [16]:
# Import all necessary libraries
import numpy as np
import os
import xarray as xr
import cfgrib
import shutil
from datetime import datetime, timedelta
import zipfile
import pandas as pd
from google.colab import files
import requests
from io import BytesIO
import csv # Added for reading CSV parameter files

# Function to load parameter specifications from a CSV file.
# This function returns raw data that can be used to augment PARAM_SPECS later.
def load_param_specs_from_csv(filepath: str) -> list:
    """
    Loads parameter specifications from a CSV file.
    Assumes CSV format: JMV_KEY,GRIB_KEY,SCALE,OFFSET,UNITS (no header).
    Returns a list of dictionaries, each representing a parameter.
    """
    additional_params_raw = []
    try:
        with open(filepath, 'r') as f:
            reader = csv.reader(f)
            for line_num, row in enumerate(reader):
                if not row or row[0].startswith('#'): # Skip empty lines or comments
                    continue
                try:
                    if len(row) == 5:
                        jmv_key = row[0].strip()
                        grib_key = row[1].strip()
                        scale = float(row[2].strip())
                        offset = float(row[3].strip())
                        units = row[4].strip()
                        additional_params_raw.append({
                            'key': jmv_key,
                            'grib_key': grib_key,
                            'scale': scale,
                            'offset': offset,
                            'units': units
                        })
                    else:
                        print(f"Warning: Skipping malformed line {line_num+1} in {filepath}: {row}. Expected 5 columns.")
                except ValueError as ve:
                    print(f"Warning: Skipping line {line_num+1} in {filepath} due to data type error: {row} - {ve}")
        print(f"Successfully read raw parameters from {filepath}.")
    except FileNotFoundError:
        print(f"Error: Parameter file not found at {filepath}")
    except Exception as e:
        print(f"An error occurred while loading parameters from {filepath}: {e}")
    return additional_params_raw

print("Libraries imported successfully.")
print("A function `load_param_specs_from_csv` has been defined to read additional parameters from '/content/GRIB2JMVALPHA.txt'.")

Libraries imported successfully.
A function `load_param_specs_from_csv` has been defined to read additional parameters from '/content/GRIB2JMVALPHA.txt'.


## 2. Core Conversion Logic (Backend)

The cells below contain the core functions that perform the GRIB2 to JMV conversion. You **do not need to modify** these cells. Simply run them to define the necessary logic for the notebook to work.

In [44]:
from __future__ import annotations

import os
from dataclasses import dataclass

import numpy as np
import pandas as pd

# xarray / cfgrib are imported lazily inside the functions that need them so
# this module can be imported (and the JMV writer unit-tested) on a machine
# without eccodes installed.


# --------------------------------------------------------------------------- #
# Parameter specifications (FNMOC conventions -- preserved from the original)  #
# --------------------------------------------------------------------------- #
@dataclass
class ParamSpec:
    """One output product. ``grib_tag`` is the eccodes shortName used to pull
    the variable out of the GRIB2 file."""
    grib_tag: str
    product_title: str
    grib_parameter_id: int
    units_code: int
    sort_level: int = 100
    process_id: int = 110
    data_type_code: int = 7
    grib_modifier_type: int = 1
    grib_modifier: float = 1.0
    data_multiplier: float = 0.1
    kelvin_to_celsius: bool = False
    missing_value: float = 0.0


PARAM_SPECS: dict[str, ParamSpec] = {
    "tmp":  ParamSpec("2t",   "2 METRE TEMPERATURE",        11, 11,  sort_level=100,
                      data_multiplier=0.1, kelvin_to_celsius=True),
    "gust": ParamSpec("gust", "WIND GUST",                 180, 180, sort_level=112,
                      data_multiplier=0.1),
    "msl":  ParamSpec("msl",  "MEAN SEA LEVEL PRESSURE",     1, 1,   sort_level=100,
                      data_multiplier=0.1),
    "10u":  ParamSpec("10u",  "10 METRE U WIND COMPONENT",  33, 33,  sort_level=100,
                      data_multiplier=0.1),
    "10v":  ParamSpec("10v",  "10 METRE V WIND COMPONENT",  34, 34,  sort_level=100,
                      data_multiplier=0.1),
    "mwd":  ParamSpec("mwd",  "MEAN WAVE DIRECTION",       107, 107, sort_level=100,
                      data_multiplier=0.1),
    "mwp":  ParamSpec("mwp",  "MEAN WAVE PERIOD",          108, 108, sort_level=100,
                      data_multiplier=0.1),
    "swh":  ParamSpec("swh",  "SIGNIFICANT WAVE HEIGHT",   100, 100, sort_level=100,
                      data_multiplier=0.01),
}

# shortName -> spec reverse lookup (used by the bundle converter)
SPECS_BY_TAG: dict[str, ParamSpec] = {s.grib_tag: s for s in PARAM_SPECS.values()}


# --------------------------------------------------------------------------- #
# JMV header + writer (no eccodes required)                                   #
# --------------------------------------------------------------------------- #
HEADER_TERMINATOR = "END_OF_HEADER"


def _encode_header(header: dict) -> bytes:
    lines = [f"{k} = {v}" for k, v in header.items()]
    return ("\n".join(lines) + f"\n{HEADER_TERMINATOR}\n").encode("ascii")


def write_jmv(file_path: str, header: dict, data: np.ndarray,
              missing_value: float = 0.0) -> str:
    """Write a 2-D grid + ASCII header to a JMV file.

    The header's ``DATA_OFFSET`` is filled in with the real byte offset of the
    grid (two-pass: the placeholder and the final value are both 5-char
    zero-padded, so the header length is stable).
    """
    data = np.asarray(data, dtype=float)
    if data.ndim == 3:
        data = data.squeeze()
    if data.ndim != 2:
        raise ValueError(f"Data must be 2-D after squeeze, got shape {data.shape}")

    grib_modifier = float(header.get("GRIB_MODIFIER", 1.0)) or 1.0
    data_multiplier = float(header.get("DATA_MULTIPLIER", 1.0)) or 1.0

    # Replace NaNs before integer cast (prevents INT_MIN overflow artifacts).
    clean = np.nan_to_num(data, nan=float(missing_value))
    scaled = np.round((clean * grib_modifier) / data_multiplier)

    # Explicit little-endian int32 so output is byte-for-byte reproducible
    # regardless of host architecture.
    grid_bytes = scaled.astype("<i4").tobytes()

    header["NUMBER_OF_RECORDS"] = str(data.size)
    header["DATA_OFFSET"] = "00000"                      # placeholder, 5 chars
    header["DATA_OFFSET"] = str(len(_encode_header(header))).zfill(5)
    final_header = _encode_header(header)

    with open(file_path, "wb") as f:
        f.write(final_header)
        f.write(grid_bytes)
    return file_path


def _dtg_and_tau(da) -> tuple[str, int]:
    """Run DTG (YYYYMMDDHHMM) + tau (hours) from a DataArray.

    Robust to both scalar coords (single-message file) and 0-d slices taken
    from a multi-step bundle via ``.isel(step=i)``.
    """
    dtg = pd.Timestamp(np.asarray(da.time.values).item()).strftime("%Y%m%d%H%M")
    if "step" in da.coords:
        tau = int(pd.Timedelta(np.asarray(da.step.values).item()).total_seconds() // 3600)
    else:
        tau = 0
    return dtg, tau


def build_header(spec: ParamSpec, da, grib_file_path: str,
                 model_tag: str = "ECMWF") -> tuple[dict, np.ndarray]:
    """Assemble the JMV ASCII header for one parameter from its DataArray."""
    native = da.values.astype(float)
    if native.ndim > 2:
        native = native.squeeze()
    # Guard: only convert K->C when values are plausibly Kelvin (mean > 200)
    if spec.kelvin_to_celsius and np.nanmean(native) > 200:
        native = native - 273.15

    dtg, tau = _dtg_and_tau(da)
    lat_n = int(da.sizes["latitude"])
    lon_n = int(da.sizes["longitude"])
    lat_spacing = abs(float(da.latitude[1] - da.latitude[0])) if lat_n > 1 else 0.25
    lon_spacing = abs(float(da.longitude[1] - da.longitude[0])) if lon_n > 1 else 0.25

    return {
        "VERSION": "1.0",
        "DATA_OFFSET": "0",
        "PLATFORM": "PC",
        "PRODUCT_TITLE": spec.product_title,
        "DATA_BASE_TITLE": spec.product_title,
        "CENTER_ID": "58",
        "PROCESS_ID": str(spec.process_id),
        "PRODUCT_TYPE": "GD",
        "UNKNOWN_PRODUCT_CODE": "0",
        "SORT_LEVEL": str(spec.sort_level),
        "DATA_TYPE_CODE": str(spec.data_type_code),
        "LABEL_CENTER_VALUE": "1",
        "GRIB_FILE_NAME": os.path.basename(grib_file_path),
        "GRIB_PARAMETER_ID": str(spec.grib_parameter_id),
        "GRIB_UNITS_CODE": "0",
        "UNITS_CODE": str(spec.units_code),
        "DATE_TIME_GROUP": dtg,
        "REQUESTED_TAU": str(tau),
        "DELIVERED_TAU": str(tau),
        "LEVEL_INDICATOR": "1",
        "LEVEL": "0.000000",
        "STANDARD_HEIGHT": "0.000000",
        "MB_LEVEL": "0.000000",
        "MODEL": model_tag,
        "BASE_LONGITUDE": f"{float(da.longitude.min()):.6f}",
        "BOTTOM_LATITUDE": f"{float(da.latitude.min()):.6f}",
        "PROJECTION": "1",
        "POLE_ON_SCREEN": "0",
        "LAT_POINTS": str(lat_n),
        "LON_POINTS": str(lon_n),
        "LABEL_LENGTH_CODE": "0",
        "HIGH_LOW_FLAG": "0",
        "TITLE_TYPE": "0",
        "DATA_MAX_VALUE": f"{np.nanmax(native):.1f}",
        "DATA_MIN_VALUE": f"{np.nanmin(native):.1f}",
        "ORIG_DATA_MAX_VALUE": f"{np.nanmax(native):.6f}",
        "ORIG_DATA_MIN_VALUE": f"{np.nanmin(native):.6f}",
        "CONTOUR_ORIGIN": "3",
        "CONTOUR_INTERVAL": "3",
        "CONTOUR_INTERVAL_COMPUTED": "NO",
        "CONTOUR_HIGH": "9999",
        "GRIB_MODIFIER_TYPE": str(spec.grib_modifier_type),
        "GRIB_MODIFIER": f"{spec.grib_modifier:.6f}",
        "DATA_MULTIPLIER": f"{spec.data_multiplier:.6f}",
        "LAND_SEA_FLAG": "1",
        "LATITUDE_GRID_SPACING": f"{lat_spacing:.6f}",
        "LONGITUDE_GRID_SPACING": f"{lon_spacing:.6f}",
        "PARTS_PER_RECORD": "1",
        "BYTES_PER_RECORD": "4",
        "RECORD_TYPE_PART1": "INTEGER",
        "BYTES_PER_POINT_PART1": "4",
    }, native


def _jmv_filename(spec: ParamSpec, da, lon_n: int, lat_n: int,
                  model_tag: str = "ECMWF") -> str:
    """Model tag now parameterized (was hardcoded 'NCEP_GFS' while the header
    said MODEL=ECMWF -- mismatch fixed)."""
    dtg, tau = _dtg_and_tau(da)
    safe = spec.product_title.replace(" ", "_")
    return f"{safe}^GD^{model_tag}_{lon_n}X{lat_n}^{dtg}^0^{tau}.JMV"


# --------------------------------------------------------------------------- #
# GRIB reading (requires eccodes / cfgrib)                                     #
# --------------------------------------------------------------------------- #
def _open_param(grib_file_path: str, short_name: str):
    """Open a single variable from a GRIB2 file by shortName, or return None."""
    import xarray as xr
    try:
        ds = xr.open_dataset(
            grib_file_path,
            engine="cfgrib",
            backend_kwargs={
                "filter_by_keys": {"shortName": short_name},
                "indexpath": "",
            },
        )
    except Exception:
        return None
    var_names = list(ds.data_vars)
    if not var_names:
        ds.close()
        return None
    return ds, ds[var_names[0]]


def detect_parameters(filepath):
    """Detect GRIB2 parameters. Return (params_list, error_msg_or_None)."""
    import cfgrib
    params = []
    try:
        with cfgrib.open_datasets(filepath) as dss:
            for ds in dss:
                params.extend(list(ds.data_vars.keys()))
    except Exception as e:
        error_msg = str(e)
        if "Edition not supported" in error_msg:
            return [], "File is not GRIB2 format (Edition 2). Check the file type."
        elif "corrupted" in error_msg.lower():
            return [], "File appears corrupted. Try re-downloading it."
        else:
            return [], f"Cannot read file: {error_msg}"
    return params, None


def _write_one(spec: ParamSpec, da2d, grib_name: str, output_dir: str,
               model_tag: str) -> str:
    """Header + JMV write for one 2-D field. Shared by both converters."""
    header, native = build_header(spec, da2d, grib_name, model_tag=model_tag)
    lat_n, lon_n = int(da2d.sizes["latitude"]), int(da2d.sizes["longitude"])
    fname = _jmv_filename(spec, da2d, lon_n, lat_n, model_tag=model_tag)
    out_path = os.path.join(output_dir, fname)
    write_jmv(out_path, header, native, missing_value=spec.missing_value)
    return out_path


def convert_grib_to_jmv(grib_file_path: str, output_dir: str,
                        param_specs: dict[str, ParamSpec] | None = None,
                        model_tag: str = "ECMWF",
                        progress=None) -> list[str]:
    """Convert every known parameter found in ``grib_file_path`` to JMV files.

    Handles multi-step files natively: if the variable carries a ``step``
    dimension, one JMV is written per forecast step. Used by Method B
    (uploaded files) and as the generic entry point.
    """
    specs = param_specs or PARAM_SPECS
    os.makedirs(output_dir, exist_ok=True)
    written: list[str] = []
    items = list(specs.items())

    for i, (key, spec) in enumerate(items):
        if progress:
            progress(spec.product_title, i, len(items))
        opened = _open_param(grib_file_path, spec.grib_tag)
        if opened is None:
            continue
        ds, da = opened
        try:
            if "step" in da.dims:                     # multi-step bundle
                for s in range(da.sizes["step"]):
                    written.append(_write_one(spec, da.isel(step=s),
                                              grib_file_path, output_dir, model_tag))
            else:                                     # single message
                written.append(_write_one(spec, da, grib_file_path,
                                          output_dir, model_tag))
        except Exception as exc:  # one bad parameter shouldn't kill the batch
            print(f"  [!] {spec.grib_tag}: {exc}")
        finally:
            ds.close()

    if progress:
        progress("done", len(items), len(items))
    return written


def convert_bundle_to_jmv(bundle_path: str, grib_tags: list[str],
                          output_dir: str, model_tag: str = "ECMWF") -> int:
    """Streamlined converter for downloaded multi-step / multi-param bundles.

    Opens each shortName ONCE (cfgrib exposes forecast steps as a ``step``
    dimension) and writes one JMV per step. Replaces the old flow of:
    binary-splitting the bundle -> writing each message to /tmp -> re-opening
    and re-indexing the file 8x per message. Returns count of JMVs written.
    """
    os.makedirs(output_dir, exist_ok=True)
    n_written = 0
    for tag in grib_tags:
        spec = SPECS_BY_TAG.get(tag)
        if spec is None:
            print(f"  [!] No ParamSpec for shortName '{tag}', skipping.")
            continue
        opened = _open_param(bundle_path, tag)
        if opened is None:
            print(f"  [!] '{tag}' not present in bundle.")
            continue
        ds, da = opened
        try:
            n_steps = da.sizes.get("step", 1)
            for s in range(n_steps):
                da2d = da.isel(step=s) if "step" in da.dims else da
                try:
                    _write_one(spec, da2d, bundle_path, output_dir, model_tag)
                    n_written += 1
                except Exception as exc:
                    print(f"  [!] {tag} step-index {s}: {exc}")
        finally:
            ds.close()
    return n_written


def preview_grid(grib_file_path: str, param_key: str, max_dim: int = 180) -> dict:
    """Downsample one parameter to a small 2-D array for the UI heatmap.

    Returns ``{values, nx, ny, vmin, vmax, title}`` (values row-major, north-up).
    """
    spec = PARAM_SPECS[param_key]
    opened = _open_param(grib_file_path, spec.grib_tag)
    if opened is None:
        raise ValueError(f"{spec.product_title} not present in file")
    ds, da = opened
    try:
        arr = da.values.astype(float)
        if arr.ndim > 2:
            arr = arr.squeeze()
        if spec.kelvin_to_celsius and np.nanmean(arr) > 200:
            arr = arr - 273.15
        # north-up: GRIB latitudes usually descend (90 -> -90); flip if ascending
        if float(da.latitude[0]) < float(da.latitude[-1]):
            arr = arr[::-1, :]
        ny, nx = arr.shape
        sy = max(1, ny // max_dim)
        sx = max(1, nx // max_dim)
        small = arr[::sy, ::sx]
        finite = small[np.isfinite(small)]
        vmin = float(np.percentile(finite, 2)) if finite.size else 0.0
        vmax = float(np.percentile(finite, 98)) if finite.size else 1.0
        return {
            "values": np.nan_to_num(small, nan=vmin).round(3).flatten().tolist(),
            "nx": int(small.shape[1]),
            "ny": int(small.shape[0]),
            "vmin": vmin,
            "vmax": vmax,
            "title": spec.product_title,
        }
    finally:
        ds.close()

print("Core conversion logic loaded (streamlined bundle converter ready).")


## 3. Choose Your Data Source

Now it's time to get your GRIB2 data! You have two main options:

*   **Method A: Download GRIB2 Data Automatically** - This option allows you to specify a data source (like GFS) and a date range, and the notebook will attempt to download the corresponding GRIB2 files for you.
*   **Method B: Upload Your Own GRIB2 Files** - If you already have GRIB2 files on your computer, you can upload them in a ZIP archive, and the notebook will process them.

**Please choose and run ONLY ONE of the following methods (Method A or Method B).**

### Method A: Download GRIB2 Data Automatically (Streamlined)

This section pulls data from the ECMWF open-data API using **bundled requests**: all parameters that share a forecast-step list are retrieved in a *single* API call per date (typically 1–2 calls total instead of one per parameter). The downloaded bundle is then converted directly to JMV — each parameter is opened once and every forecast step is written out — with no intermediate file splitting.

Configure the form below, then run the cell.

In [52]:
#@title Streamlined ECMWF Open-Data Downloader { display-mode: "form" }
#@markdown One bundled API request per (date, step-group) instead of one per
#@markdown parameter -- then a single cfgrib open per parameter for conversion.

from datetime import datetime, timedelta
import os
import logging
from ecmwf.opendata import Client

logging.getLogger("multiurl").setLevel(logging.ERROR)
logging.getLogger("urllib3").setLevel(logging.ERROR)

#@markdown #### 1. Date Range Settings
ecmwf_start_date_str = '2026-07-13' #@param {type:"date"}
ecmwf_end_date_str = '2026-07-13' #@param {type:"date"}

#@markdown #### 2. Model Settings & Streams
ecmwf_model_type = 'HRES' #@param ["HRES", "AIFS", "ENSEMBLE", "WAVE"]
ecmwf_model_cycle = '00' #@param ["00", "06", "12", "18"]
ecmwf_resolution = "0.25/0.25" #@param ["0.25/0.25", "0.4/0.4", "0.5/0.5", "1.0/1.0"]

#@markdown #### 3. Forecast Steps (Temporal Settings)
ecmwf_max_forecast_hour = 240 #@param {type:"slider", min:0, max:240, step:24}
ecmwf_finest_forecast_interval = 6 #@param {type:"slider", min:1, max:24, step:1}

# --- Model mapping specs -------------------------------------------------
# ``client_model`` selects the correct ecmwf-opendata backend. NOTE: the old
# version always used the default Client(), which serves IFS -- so selecting
# AIFS silently downloaded IFS HRES data. Fixed here.
MODEL_CONFIGS = {
    "HRES":     {"stream": "oper", "type": "fc", "client_model": "ifs",
                 "parameters": ["2t", "10u", "10v", "msl", "gust"],
                 "valid_cycles": ["00", "06", "12", "18"]},
    "AIFS":     {"stream": "oper", "type": "fc", "client_model": "aifs-single",
                 "parameters": ["2t", "10u", "10v", "msl"],
                 "valid_cycles": ["00", "06", "12", "18"]},
    "ENSEMBLE": {"stream": "enfo", "type": "cf", "client_model": "ifs",
                 "parameters": ["2t", "10u", "10v", "msl", "gust", "swh"],
                 "valid_cycles": ["00", "06", "12", "18"]},
    "WAVE":     {"stream": "wave", "type": "fc", "client_model": "ifs",
                 "parameters": ["swh", "mwd", "mwp"],
                 "valid_cycles": ["00", "12"]},
}

config = MODEL_CONFIGS[ecmwf_model_type]

# --- Config validation / auto-correction ---------------------------------
if ecmwf_model_cycle not in config["valid_cycles"]:
    corrected = config["valid_cycles"][0]
    print(f"[!] {ecmwf_model_type} is not run at {ecmwf_model_cycle}z. "
          f"Shifting to {corrected}z.")
    ecmwf_model_cycle = corrected

if ecmwf_model_type == "AIFS" and ecmwf_resolution != "0.25/0.25":
    print("[!] AIFS open data is only published at 0.25 deg. Forcing 0.25/0.25.")
    ecmwf_resolution = "0.25/0.25"

output_jmv_ecmwf_dir = '/content/jmv_output_ecmwf'
os.makedirs(output_jmv_ecmwf_dir, exist_ok=True)

forecast_steps = sorted(set(range(0, ecmwf_max_forecast_hour + 1,
                                  ecmwf_finest_forecast_interval)) | {0})

# Only request parameters we can actually convert (have a ParamSpec)
requested_tags = [t for t in config["parameters"] if t in SPECS_BY_TAG]

# --- Group parameters by their valid step list ---------------------------
# gust has no analysis (step 0) field, so it needs its own step list. All
# params sharing a step list go into ONE bundled retrieve -> typically 1-2
# API calls per date instead of one per parameter.
step_groups: dict[tuple, list[str]] = {}
for tag in requested_tags:
    steps = tuple(s for s in forecast_steps if s != 0) if tag == "gust" \
        else tuple(forecast_steps)
    step_groups.setdefault(steps, []).append(tag)

client = Client(source="ecmwf", model=config["client_model"])

# --- Availability probe: skip dates the open-data rolling window no longer
# serves, instead of failing per-request. -----------------------------------
try:
    latest_run = client.latest(type=config["type"], stream=config["stream"])
    print(f"Latest available {ecmwf_model_type} run on server: {latest_run}")
except Exception as probe_err:
    latest_run = None
    print(f"[i] Availability probe skipped ({probe_err})")

print("\n=== [METOC Planning Tool] Streamlined Downloader ===")
print(f"Model / Cycle  : {ecmwf_model_type} @ {ecmwf_model_cycle}z "
      f"(client model: {config['client_model']})")
print(f"Date Range     : {ecmwf_start_date_str} to {ecmwf_end_date_str}")
print(f"Resolution     : {ecmwf_resolution}")
print(f"Steps          : {len(forecast_steps)} "
      f"(0..{ecmwf_max_forecast_hour}h @ {ecmwf_finest_forecast_interval}h)")
print(f"Parameters     : {requested_tags}")
print(f"API calls/date : {len(step_groups)} bundled retrieve(s)")

start_date = datetime.strptime(ecmwf_start_date_str, '%Y-%m-%d').date()
end_date = datetime.strptime(ecmwf_end_date_str, '%Y-%m-%d').date()

total_written = 0
current_date = start_date
while current_date <= end_date:
    ecmwf_model_run_date_str = current_date.strftime('%Y-%m-%d')
    print(f"\n--- {ecmwf_model_run_date_str} {ecmwf_model_cycle}z ---")

    date_written = 0
    for steps, tags in step_groups.items():
        bundle_name = (f"ecmwf_{ecmwf_model_type.lower()}_"
                       f"{ecmwf_model_run_date_str.replace('-', '')}_"
                       f"{ecmwf_model_cycle}_{'-'.join(tags)}.grib2")
        bundle_path = os.path.join('/tmp', bundle_name)

        def _retrieve(params):
            client.retrieve(
                date=ecmwf_model_run_date_str,
                time=ecmwf_model_cycle,
                type=config["type"],
                step=list(steps),
                param=params,
                stream=config["stream"],
                grid=ecmwf_resolution,
                target=bundle_path,
            )

        try:
            print(f"Retrieving bundle: {tags} x {len(steps)} steps "
                  f"(single request)...")
            try:
                _retrieve(tags)                    # bundled multi-param request
                convert_tags = tags
            except Exception as bundle_err:
                # One unavailable param can 404 the whole bundle -- fall back
                # to per-param retrieval so the rest still convert.
                print(f"  [!] Bundled request failed ({bundle_err}); "
                      f"falling back to per-parameter retrieval.")
                convert_tags = []
                for tag in tags:
                    try:
                        single_path = bundle_path + f".{tag}"
                        client.retrieve(
                            date=ecmwf_model_run_date_str,
                            time=ecmwf_model_cycle,
                            type=config["type"], step=list(steps), param=tag,
                            stream=config["stream"], grid=ecmwf_resolution,
                            target=single_path)
                        n = convert_bundle_to_jmv(single_path, [tag],
                                                  output_jmv_ecmwf_dir,
                                                  model_tag=ecmwf_model_type)
                        date_written += n
                        os.remove(single_path)
                    except Exception as p_err:
                        print(f"    [ERROR] {tag}: {p_err}")
                continue  # per-param path already converted

            # Convert straight from the bundle: one cfgrib open per param,
            # one JMV per (param, step). No splitting, no /tmp step files.
            n = convert_bundle_to_jmv(bundle_path, convert_tags,
                                      output_jmv_ecmwf_dir,
                                      model_tag=ecmwf_model_type)
            date_written += n
            print(f"  --> {n} JMV files written from bundle.")

        except Exception as e:
            print(f"  [ERROR] {tags}: {e}")
        finally:
            if os.path.exists(bundle_path):
                os.remove(bundle_path)

    print(f"Date complete: {date_written} JMV files.")
    total_written += date_written
    current_date += timedelta(days=1)

print(f"\n=== Done. {total_written} JMV files in {output_jmv_ecmwf_dir} ===")



=== [METOC Planning Tool] High-Speed Downloader Verified ===
Model Selected : AIFS
Model Cycle    : 18z
Date Range     : 2026-07-15 to 2026-07-13
Resolution     : 0.25/0.25
API Mode       : Bundled (High-Speed Single Stream/Parameter Retrieval)
Active parameters for download: ['2t', 'gust', 'msl', '10u', '10v']


### Method B: Upload Your Own GRIB2 Files

If you have your own GRIB2 files, you can upload them as a single `.zip` archive. The notebook will then extract and convert all GRIB2 files found within the archive.

Run the cell below to create a temporary directory on the cloud, When this session is over, all data will be removed, so ensure that you export what you need.

Create a Temporary directory by running the cell below, once completed, upload the products to said directory to download the zipped package.













In [ ]:
# Create a temporary directory for uploads
upload_dir = '/content/grib2_uploads'
os.makedirs(upload_dir, exist_ok=True)

print("Please upload a .zip file containing your GRIB2 files using the button below.")

uploaded = files.upload()

if uploaded:
    for fn in uploaded.keys():
        print(f'User uploaded file "{fn}"')
        # Save the uploaded file to the upload directory
        with open(os.path.join(upload_dir, fn), 'wb') as f:
            f.write(uploaded[fn])
    print("Upload complete. Proceed to the next cell to unzip and convert.")
else:
    print("No files were uploaded.")

In [43]:
import os
import shutil

#@title Clear All Generated JMV and Uploaded GRIB2 Files { display-mode: "form" }

# Define the directories to clear
jmv_ecmwf_dir = '/content/jmv_output_ecmwf'
jmv_local_dir = '/content/jmv_output_local'
grib2_uploads_dir = '/content/grib2_uploads'
tmp_dir = '/tmp' # Temporary directory used for grib2 files during ECMWF download

def clear_directory(path):
    if os.path.exists(path):
        try:
            shutil.rmtree(path)
            print(f"Successfully cleared and removed: {path}")
        except OSError as e:
            print(f"Error clearing directory {path}: {e}")
    else:
        print(f"Directory not found, no action needed: {path}")

choice = input("Clear all data? Y/N").strip().lower()

if choice == "y":

  print("Initiating cleanup...")

  clear_directory(jmv_ecmwf_dir)
  clear_directory(jmv_local_dir)
  clear_directory(grib2_uploads_dir)

  # Clear temporary GRIB2 files from /tmp
  # It's safer to only remove files matching a pattern rather than the entire /tmp directory
  print(f"Cleaning temporary GRIB2 files from {tmp_dir}...")
  removed_count = 0
  for filename in os.listdir(tmp_dir):
      if filename.startswith('ecmwf_') and (filename.endswith('.grib2') or filename.endswith('.grib')):
          file_path = os.path.join(tmp_dir, filename)
          try:
              os.remove(file_path)
              removed_count += 1
          except OSError as e:
              print(f"Error removing temporary file {file_path}: {e}")
  print(f"Removed {removed_count} temporary GRIB2 files from {tmp_dir}.")

  print("Cleanup complete. All specified output and temporary directories/files have been cleared.")
elif choice == "n":
  print("Cleanup aborted. No files were removed.")
else:
  print("Invalid choice. Cleanup aborted. Please enter 'Y' or 'N'.")

Clear all data? Y/Ny
Initiating cleanup...
Successfully cleared and removed: /content/jmv_output_ecmwf
Directory not found, no action needed: /content/jmv_output_local
Directory not found, no action needed: /content/grib2_uploads
Cleaning temporary GRIB2 files from /tmp...
Removed 0 temporary GRIB2 files from /tmp.
Cleanup complete. All specified output and temporary directories/files have been cleared.


In [48]:
# Unzip uploaded archive(s) and convert every GRIB2 found (Method B)
upload_dir = '/content/grib2_uploads/'
output_jmv_local_dir = '/content/jmv_output_local/'
os.makedirs(output_jmv_local_dir, exist_ok=True)

zip_files = sorted(f for f in os.listdir(upload_dir) if f.endswith('.zip'))

if not zip_files:
    print("No .zip file found in the upload directory. "
          "Please upload a .zip file first (Method B).")
else:
    # BUGFIX: old version set zip_filepath for multiple zips but then never
    # unzipped (extraction lived in the else branch). Now all zips process.
    if len(zip_files) > 1:
        print(f"Multiple .zip files found ({len(zip_files)}); processing all.")

    for zip_name in zip_files:
        zip_filepath = os.path.join(upload_dir, zip_name)
        print(f"Unzipping: {zip_name}")
        try:
            with zipfile.ZipFile(zip_filepath, 'r') as zip_ref:
                zip_ref.extractall(upload_dir)
        except zipfile.BadZipFile:
            print(f"  [!] {zip_name} is not a valid ZIP file, skipping.")

    grib2_files_found = 0
    jmv_written = 0
    for root, _, files_in_dir in os.walk(upload_dir):
        for file_name in files_in_dir:
            if file_name.endswith(('.grib2', '.grb2', '.grib')):
                grib_filepath = os.path.join(root, file_name)
                print(f"Converting {file_name}...")
                try:
                    written = convert_grib_to_jmv(grib_filepath,
                                                  output_jmv_local_dir)
                    jmv_written += len(written)
                    grib2_files_found += 1
                except Exception as e:
                    print(f"  [!] Failed on {file_name}: {e}")

    if grib2_files_found == 0:
        print("No GRIB2 files found in the uploaded archive(s). Ensure your "
              "ZIP contains .grib2, .grb2, or .grib files.")
    else:
        print(f"Conversion complete: {grib2_files_found} GRIB2 files -> "
              f"{jmv_written} JMV files.")

print("Local GRIB2 upload and conversion attempt finished.")


FileNotFoundError: [Errno 2] No such file or directory: '/content/grib2_uploads/'

## 4. Package and Download Your JMV Files

The final step is to collect all the generated JMV files and package them into a single `.zip` archive. This archive can then be easily downloaded to your local computer.

In [ ]:
#@markdown ### Select the folder containing your converted JMV files
#@markdown Choose the output directory based on which method (A or B) you used for conversion.
output_folder_to_package = '/content/jmv_output_ecmwf' #@param ["/content/jmv_output_ecmwf", "/content/jmv_output_local"]

print(f"Selected output folder for packaging: {output_folder_to_package}")

In [50]:
# Package the selected output folder and download it
from datetime import datetime as _dt

# Metadata from the downloader if Method A was run; sane fallbacks otherwise
_run_date = globals().get('ecmwf_model_run_date_str',
                          _dt.utcnow().strftime('%Y-%m-%d'))
_cycle = globals().get('ecmwf_model_cycle', '00')
_model = globals().get('ecmwf_model_type', 'LOCAL')

formatted_date = _run_date.replace('-', '')
formatted_time = f"{_cycle}00"

zip_filename = f"Converted_JMV_Files_{_model}_{formatted_date}-{formatted_time}"
zip_path = '/content/' + zip_filename

if os.path.isdir(output_folder_to_package) and len(os.listdir(output_folder_to_package)) > 0:
    try:
        shutil.make_archive(zip_path, 'zip', output_folder_to_package)
        print(f"Successfully created zip file: {zip_path}.zip")
        files.download(zip_path + '.zip')
        print("Download initiated. Check your browser's downloads.")
    except Exception as e:
        print(f"Error during zipping or downloading: {e}")
else:
    print(f"The selected folder '{output_folder_to_package}' does not exist or is empty. "
          "Run one of the conversion methods (A or B) first.")


Successfully created zip file: /content/Converted_JMV_Files_AIFS_20260713-0000.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download initiated. Check your browser's downloads.
